In [1]:
import os
import time
import torch
from langchain.document_loaders import PyPDFLoader, WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain.llms.base import LLM
from typing import Optional, List, Any

# Environment Setup (Using GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set Seed for Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

USER_AGENT environment variable not set, consider setting it to identify your requests.


Using device: cuda


# Task 1: Source Discovery
## 1) Relevant sources

In [2]:
references_dir = "References/"
file_names = [f for f in os.listdir(references_dir) if os.path.isfile(os.path.join(references_dir, f))]
pdf_files = [os.path.join(references_dir, file) for file in file_names]
print(pdf_files)

['References/A6_Bio.pdf']


In [ ]:
web_links = []

# Load and process documents
def load_documents(pdf_paths):
    documents = []
    from langchain.document_loaders import PyPDFLoader, WebBaseLoader

    for pdf in pdf_paths:
        loader = PyPDFLoader(pdf)
        documents.extend(loader.load())

    web_loader = WebBaseLoader(web_links)
    documents.extend(web_loader.load())

    return documents

# Loading documents
documents = load_documents(pdf_files, web_links)

# Splitting documents for embeddings
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(documents)

## 2) Prompt template

In [4]:
# Define Prompt
prompt_template = """
Answer the following question very shortly(no more than 50 words) based solely on the provided context.
If the answer is not present in the context, say 'I don't know.'

Context:
{context}

Question:
{input}

Answer clearly:
""".strip()

prompt = ChatPromptTemplate.from_template(template = prompt_template)

# Task 2: Analysis and Problem Solving
## Define retriever

- Embeddings and Vector Store
- Embedding Model: Using **sentence-transformers/all-MiniLM-L6-v2**
 for efficient retrieval.
- This model provides fast and accurate embeddings suitable for question-answering tasks.
- FAISS is used for storing and retrieving document embeddings.

In [5]:
# Embeddings and Vector Store
embedding_model = HuggingFaceEmbeddings(
    model_name= 'sentence-transformers/all-MiniLM-L6-v2',
    # model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={"device": device}
)

vector_store = FAISS.from_documents(split_documents, embedding_model)

In [6]:
# Store vector locally
vector_path = 'vector-store'
db_file_name = 'nlp_vector_store'

vector_store.save_local(
    folder_path = os.path.join(vector_path, db_file_name),
    index_name = 'nlp' #default index
)

In [7]:
# vector_store = FAISS.load_local(vector_store_folder, embeddings=embedding_model)

In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

### Retriever model testing

In [9]:
sample_query = "How do you think cultural values should influence technological advancements?"
retrieved_docs = retriever.get_relevant_documents(sample_query)
print(f"Retrieved information for query '{sample_query}':\n")
for i, doc in enumerate(retrieved_docs):
    print(f"Doc {i+1}: {doc.page_content[:200]}\n")

Retrieved information for query 'How do you think cultural values should influence technological advancements?':

Doc 1: ethics, needs, and priorities. However, this influence must be balanced to avoid stifling 
progress. Below, I’ll outline how cultural values should impact technology and why this 
balance matters. 
 


Doc 2: techniques. 
 
# Core Beliefs on Technology 
I strongly believe that technology should be developed and applied ethically, ensuring 
inclusivity, fairness, and positive societal impact. AI must be des

Doc 3: - Developing AI-driven tools to assist medical professionals in disease diagnosis. 
- Exploring transformer-based models to improve the interpretability of medical text data. 
- Enhancing patient care



/tmp/ipykernel_60435/4211253318.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(sample_query)


## Generator Model Setup

### Best Model exploration

In [10]:
generator_models = [
    "microsoft/Phi-4-mini-instruct",
    "mistralai/Mistral-7B-Instruct-v0.2",
    "Qwen/Qwen2.5-Coder-0.5B-Instruct"     
]

def test_models(model_name):
    start_time = time.time()  # Start timer
    print(f"\nTesting model: {model_name}")
    
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        device_map="auto", 
        torch_dtype=torch.float16
    )

    # Initialize pipeline
    llm_pipeline = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.1,
        repetition_penalty=1.2
    )

    # Test the model with a sample question
    test_question = "What is your highest level of education?\n"
    
    # Time inference separately (optional)
    inference_start = time.time()
    response = llm_pipeline(test_question)[0]['generated_text']
    inference_time = time.time() - inference_start
    
    total_time = time.time() - start_time  # Total execution time
    
    print(f"Response from {model_name}: {response}\n")
    print(f"Total Runtime: {total_time:.2f}s | Inference Time: {inference_time:.2f}s\n")

In [11]:
# Run tests on all models
for model in generator_models:
    test_models(model)


Testing model: microsoft/Phi-4-mini-instruct


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Response from microsoft/Phi-4-mini-instruct: What is your highest level of education?
I have a Bachelor's degree in Computer Science from the University X. I graduated with honors and received several awards for my academic performance.
My Bachelor’s Degree
A bachelor, or B.A., refers to an undergraduate college-level diploma awarded after completing four years (or six semesters) at university following high school graduation.

Bachelor's degrees are typically divided into two categories: liberal arts/bachelor-of-science/BA/BSc; professional bachelor's programs such as nursing science/nursing BA/NBSC etc.; engineering/sciences/mathematics/computer sciences/programs like yours - which usually require more coursework than other types but also provide students greater specialization opportunities within their chosen field(s).

In addition there may be some differences between universities regarding how they award these diplomas depending on whether they're public institutions vs private o

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Response from mistralai/Mistral-7B-Instruct-v0.2: What is your highest level of education?
I have a Bachelor's degree in Computer Science.
What are some of the most common challenges you face as a software engineer?
Some of the most common challenges I face as a software engineer include:
1. Debugging complex issues that can take hours or even days to resolve.
2. Keeping up with new technologies and programming languages, which can be time-consuming but necessary for staying competitive in the industry.
3. Collaborating effectively with other team members, including designers, project managers, and other developers, to ensure projects are delivered on time and meet quality standards.
4. Balancing competing priorities and deadlines while maintaining a high level of attention to detail and focus on delivering high-quality code.
5. Ensuring security and data privacy concerns are addressed in the development process.
6. Managing technical debt and ensuring codebase remains maintainable and

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Device set to use cuda:0


Response from Qwen/Qwen2.5-Coder-0.5B-Instruct: What is your highest level of education?
I am a 20-year-old student at the University of California, Berkeley. I have completed my undergraduate degree in computer science and mathematics from UC Berkeley.
Can you explain how to write an essay on this topic? How would you structure it? Would you need any additional information or resources for writing an essay on this topic?

Total Runtime: 7.71s | Inference Time: 2.91s



- *mistralai/Mistral-7B-Instruct-v0.2* has the best response.
- But due to large computational cost and long runtime, I will be using Qwen2.5-Coder-0.5B-Instruct
- This model generates responses based on retrieved information.

In [12]:
model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", torch_dtype=torch.float16
)

llm_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    repetition_penalty=1.2
)

# Custom wrapper for LangChain compatibility
class CustomPipelineLLM(LLM):
    pipeline: Any

    @property
    def _llm_type(self) -> str:
        return "custom_pipeline"

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        output = self.pipeline(prompt, max_new_tokens=256, do_sample=True, temperature=0.2)[0]['generated_text']
        return output.strip().split("Answer clearly:")[-1].strip()

llm_custom = CustomPipelineLLM(pipeline=llm_pipeline)

Device set to use cuda:0


## Save Generator Model Locally

In [13]:
# generator_model_path = "RAG/generator_model"
# tokenizer.save_pretrained(generator_model_path)
# model.save_pretrained(generator_model_path)

## Generator Model Testing

In [14]:
response = llm_pipeline("Explain the importance of NLP in healthcare",
                        max_new_tokens=256,
                        do_sample=True,
                        temperature=0.3,
                        pad_token_id=tokenizer.eos_token_id)

print("\nGenerated Response:\n", response[0]["generated_text"])


Generated Response:
 Explain the importance of NLP in healthcare and how it can be used to improve patient outcomes.

NLP (Natural Language Processing) is a subset of machine learning that enables computers to understand, interpret, generate, and perform natural language tasks. It has become increasingly important in healthcare due to its ability to analyze large amounts of text data and identify patterns and trends that may not be immediately apparent for humans.

One of the most significant benefits of using NLP in healthcare is that it can help improve patient outcomes by providing more accurate diagnoses, better treatment plans, and personalized care. For example, doctors can use NLP to analyze medical records and identify potential health issues before they are diagnosed. This information can then be used to develop targeted treatments or interventions that address specific problems.

Another benefit of using NLP in healthcare is that it can also help reduce errors and biases in 

# Task 3: Chatbot Development

In [15]:
# RAG Chain Integration
document_chain = create_stuff_documents_chain(llm_custom, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

In [16]:
# Test queries
questions = [
"How old are you?",
"What is your highest level of education?",
"What major or field of study did you pursue during your education?",
"How many years of work experience do you have?",
"What type of work or industry have you been involved in?",
"Can you describe your current role or job responsibilities?",
"What are your core beliefs regarding the role of technology in shaping society?",
"How do you think cultural values should influence technological advancements?",
"As a master’s student, what is the most challenging aspect of your studies so far?",
"What specific research interests or academic goals do you hope to achieve during your time as a master’s student?"
]

In [17]:
# Generate Responses
responses = []
for question in questions:
    result = rag_chain.invoke({"input": question})
    answer = result['answer'].strip()

    # Clean answer: Remove redundant explanations
    clean_answer = answer.split("\n")[0]
    
    print(f"Question: {question}\nAnswer: {clean_answer}\n")
    responses.append({"question": question, "answer": clean_answer})

Question: How old are you?
Answer: 28

Question: What is your highest level of education?
Answer: Bachelor's Degree

Question: What major or field of study did you pursue during your education?
Answer: AI

Question: How many years of work experience do you have?
Answer: 5 years

Question: What type of work or industry have you been involved in?
Answer: Machine Learning Engineer

Question: Can you describe your current role or job responsibilities?
Answer: Data Analyst

Question: What are your core beliefs regarding the role of technology in shaping society?
Answer: I strongly believe that technology should be developed and applied ethically, ensuring inclusivity, fairness, and positive societal impact.

Question: How do you think cultural values should influence technological advancements?
Answer: Cultural values should guide innovation in ways that reflect societal norms and expectations.



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Question: As a master’s student, what is the most challenging aspect of your studies so far?
Answer: One of the most challenging aspects of my studies is balancing theoretical coursework with real-world AI applications.

Question: What specific research interests or academic goals do you hope to achieve during your time as a master’s student?
Answer: I want to pursue NLP applications in healthcare specifically.



In [18]:
# Output JSON format
import json
output_path = "chatbot_responses.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(responses, f, ensure_ascii=False, indent=4)

print("Responses saved successfully to chatbot_responses.json")

Responses saved successfully to chatbot_responses.json
